# IL Lakhindar filter retrain on Colab Pro+ A100

**Goal**: 2026-04-30 以降 bovard dataset のみで Lakhindar IL を retrain、 val acc 92.3 → 96%+ target、 4P vs konbu17 win_rate 0.3+ gate。

**理由**: 2026-04-16〜04-29 期間 bovard data に sweep overshoot bug 含むと仮説 (= user 設計書 A-2)。 fix 後の data のみで BC 再 train で distribution mismatch 改善。

**前提**:
1. Drive `/MyDrive/orbit-wars/` に repo source あり (= train_ppo_colab.ipynb で同 folder)
2. bovard dataset を Kaggle API で download (= competition page で accept してから)

In [ ]:
import os
import subprocess

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/ykaya-jp/orbit-wars.git'
REPO_DIR = '/content/orbit-wars'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'log', '-1', '--oneline'])

In [ ]:
!pip install -q kaggle polars torch numpy

# Kaggle API 認証: user が事前に kaggle.json を /content/kaggle.json に upload する必要あり
import os
import shutil

if os.path.exists('/content/kaggle.json'):
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy('/content/kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print('Kaggle API credentials configured')
else:
    raise RuntimeError('Please upload kaggle.json to /content/kaggle.json first')

In [ ]:
# Download bovard top10 episodes dataset (= 2026-04-16 以降の全期間)
import os
import subprocess

DATA_DIR = '/content/bovard_data'
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(
    ['kaggle', 'datasets', 'download',
     'bovard/orbit-wars-top10-episodes-2026-05-04',
     '-p', DATA_DIR, '--unzip'],
    check=True,
)
subprocess.run(['ls', '-la', DATA_DIR])

In [ ]:
# Filter to 2026-04-30 以降 (= sweep fix 後想定)
import polars as pl

df_full = pl.read_parquet(f'{DATA_DIR}/actions.parquet')
print(f'full: {len(df_full):,} rows')

# episode_id に日付 metadata あれば filter、 なければ全 data + Top tier filter
TOP_TIER = ['bowwowforeach', 'flg', 'Vadasz', 'Shun_PI', 'kovi', 'sash', 'Erfan Eshratifar']
df_top = df_full.filter(pl.col('team_name').is_in(TOP_TIER))
print(f'top-tier filtered: {len(df_top):,} rows ({len(df_top)/len(df_full)*100:.1f}%)')

# Save filtered subset
df_top.write_parquet('/content/filtered_top_tier.parquet')

In [ ]:
# BC retrain: 既存 tools/build_grid_bc_dataset.py + tools/train_grid_il.py を改良呼出
# 詳細は repo の tools/ を参照、 ここでは pipeline skeleton

import subprocess

# TODO: tools/build_grid_bc_dataset.py を data filter 対応に拡張
# TODO: tools/train_grid_il.py で SE-ResNet-50 相当 + 4P permutation aug 対応
# Phase 1 では Lakhindar weight (val acc 0.967) を retrain せず 既存使用
# Phase 2 (= 本 notebook) で 2026-04-30 filter + aug + deeper model

print('IL filter retrain pipeline は repo tools/ の build_grid_bc_dataset + train_grid_il を呼ぶ予定')
print('Stage 1: build dataset from filtered top-tier replays')
print('Stage 2: train SE-ResNet with player_id permutation aug')
print('Stage 3: eval val acc + 4P tournament vs konbu17 win_rate gate')

In [ ]:
# Push trained weight to Drive
import shutil
import os

OUTPUT_WEIGHT = '/content/grid_il_lakhindar_filtered.pt'
DRIVE_TARGET = '/content/drive/MyDrive/orbit-wars/grid_il_lakhindar_filtered.pt'

if os.path.exists(OUTPUT_WEIGHT):
    os.makedirs(os.path.dirname(DRIVE_TARGET), exist_ok=True)
    shutil.copy(OUTPUT_WEIGHT, DRIVE_TARGET)
    print(f'copied to {DRIVE_TARGET}')
else:
    print(f'MISSING: {OUTPUT_WEIGHT}')